# NB_patient_360 — Gold Layer

Builds `adb_caresync.gold.patient_360`: one row per patient enriched with hospital name, insurance provider, and aggregated lab result metrics (total, abnormal, critical tests + last test date). Full rebuild on every run.

In [0]:
%sql
-- Ensure the gold schema exists before writing
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold

In [0]:
# Verify all required source silver tables are accessible
source_tables = [
    "adb_caresync.silver.patients",
    "adb_caresync.silver.hospitals",
    "adb_caresync.silver.insurance_providers",
    "adb_caresync.silver.lab_results",
]

for t in source_tables:
    count = spark.table(t).count()
    schema = spark.table(t).dtypes
    print(f"✅ {t}: {count:,} rows | columns: {[c[0] for c in schema]}")

In [0]:
%sql
-- ─────────────────────────────────────────────────────────────
-- Target : adb_caresync.gold.patient_360
-- Grain  : One row per patient
-- Sources: silver.patients (driving)
--          silver.hospitals       (LEFT JOIN on registered_hospital_id)
--          silver.insurance_providers (LEFT JOIN on insurance_provider_id)
--          silver.lab_results     (aggregated sub-query per patient)
-- Load   : FULL — CREATE OR REPLACE on every run
-- ─────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360 AS

WITH lab_agg AS (
    -- Pre-aggregate lab results per patient to avoid fan-out
    SELECT
        patient_id,
        COUNT(*)                                                   AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)    AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)    AS critical_test_count,
        MAX(test_date)                                             AS last_test_date
    FROM adb_caresync.silver.lab_results
    GROUP BY patient_id
)

SELECT
    -- ── Surrogate / Primary Key ────────────────────────────────
    p.patient_id,

    -- ── Patient Demographics ───────────────────────────────────
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                 AS date_of_birth,
    CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT)
                                                                  AS age,
    p.gender,
    p.city,
    p.state,

    -- ── Hospital (via LEFT JOIN) ────────────────────────────────
    p.registered_hospital_id                                      AS hospital_id,
    h.hospital_name,

    -- ── Insurance (via LEFT JOIN) ──────────────────────────────
    p.insurance_provider_id,
    ip.provider_name                                              AS insurance_provider_name,

    -- ── Lab Result Aggregates (via sub-query) ──────────────────
    COALESCE(la.total_tests,         0)                           AS total_tests,
    COALESCE(la.abnormal_test_count, 0)                           AS abnormal_test_count,
    COALESCE(la.critical_test_count, 0)                           AS critical_test_count,
    la.last_test_date                                             -- NULL if no tests

FROM      adb_caresync.silver.patients           p
LEFT JOIN adb_caresync.silver.hospitals          h  ON p.registered_hospital_id  = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip ON p.insurance_provider_id   = ip.insurance_provider_id
LEFT JOIN lab_agg                                la  ON p.patient_id              = la.patient_id

In [0]:
record_count = spark.table("adb_caresync.gold.patient_360").count()
dbutils.notebook.exit(str(record_count))